<a href="https://colab.research.google.com/github/Ranjith1816/Gen_AI--Assignment-_task/blob/main/Fine_Tuning_BERT_on_a_Kaggle_Dataset_ipynb_txt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

 install dependencies

In [ ]:
!pip install transformers datasets accelerate scikit-learn -q

In [ ]:
import torch
import numpy as np
import pandas as pd
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    confusion_matrix, classification_report, ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
if device.type == 'cuda':
    print("GPU is enabled 🚀")
else:
    print("Running on CPU (slower)")

2.uploading dataset

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import zipfile, os

zip_file = list(uploaded.keys())[0]

with zipfile.ZipFile(zip_file, 'r') as zip_ref:
    zip_ref.extractall('/content/data')

print(os.listdir('/content/data'))

3. loading data

In [ ]:
column_names = ['id', 'entity', 'sentiment', 'text']
train_df = pd.read_csv('/content/data/twitter_training.csv', names=column_names, header=None)
val_df = pd.read_csv('/content/data/twitter_validation.csv', names=column_names, header=None)

print(train_df.head())
print(val_df.head())

In [ ]:
print(train_df.columns)
print(train_df['sentiment'].value_counts())

4. preprocessing

In [ ]:
import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+", "", text)   # remove URLs
    text = re.sub(r"@\w+", "", text)      # remove mentions
    text = re.sub(r"#\w+", "", text)      # remove hashtags
    text = re.sub(r"[^a-z\s]", "", text)  # remove special chars
    return text

train_df['text'] = train_df['text'].apply(clean_text)
val_df['text'] = val_df['text'].apply(clean_text)

In [ ]:
train_df = train_df.dropna()
val_df = val_df.dropna()

In [ ]:
#testing set
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(train_df, test_size=0.1, random_state=42)

5. tokenization

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [ ]:
train_encodings = tokenizer(list(train_df['text']), truncation=True, padding=True)
val_encodings = tokenizer(list(val_df['text']), truncation=True, padding=True)
test_encodings = tokenizer(list(test_df['text']), truncation=True, padding=True)

In [ ]:
class Dataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels.tolist()

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

# Define the label map
label_map = {
    'Positive': 0,
    'Negative': 1,
    'Neutral': 2,
    'Irrelevant': 3
}

# Map sentiment labels to integers for all dataframes
train_df['label'] = train_df['sentiment'].map(label_map)
val_df['label'] = val_df['sentiment'].map(label_map)
test_df['label'] = test_df['sentiment'].map(label_map)

train_dataset = Dataset(train_encodings, train_df['label'])
val_dataset = Dataset(val_encodings, val_df['label'])
test_dataset = Dataset(test_encodings, test_df['label'])

6.loading BERT model

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=len(label_map))
model.to(device)

7. fine tuning

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=2,   #acts like batch size 32
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    eval_strategy="epoch", # Changed from evaluation_strategy
    logging_dir="./logs",
    fp16=True
)

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted')
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)


In [ ]:
trainer.train()

8. evaluation

In [ ]:
predictions_full_bert = trainer.predict(test_dataset)

y_pred_full_bert = np.argmax(predictions_full_bert.predictions, axis=1)
y_true_full_bert = predictions_full_bert.label_ids

print(classification_report(y_true_full_bert, y_pred_full_bert))

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_true, y_pred)
plt.show()

Experiments

In [ ]:
#freeze BERT
for param in model.bert.parameters():
    param.requires_grad = False

trainer.train()

In [ ]:
predictions_frozen_bert = trainer.predict(test_dataset)

y_pred_frozen_bert = np.argmax(predictions_frozen_bert.predictions, axis=1)
y_true_frozen_bert = predictions_frozen_bert.label_ids

print(classification_report(y_true_frozen_bert, y_pred_frozen_bert))

In [ ]:
evaluate_model("Experiment 1: (Frozen BERT)", y_true, y_pred)

In [ ]:
# unfreeze only last 2 layers
for name, param in model.bert.named_parameters():
    if "encoder.layer.10" in name or "encoder.layer.11" in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

trainer.train()

In [ ]:
predictions_2layer_bert = trainer.predict(test_dataset)

y_pred_2layer_bert = np.argmax(predictions_2layer_bert.predictions, axis=1)
y_true_2layer_bert = predictions_2layer_bert.label_ids

print(classification_report(y_true_2layer_bert, y_pred_2layer_bert))

In [ ]:
evaluate_model("Experiment 2: (Last 2 Layers)", y_true, y_pred)

9. comparision

In [ ]:
results = []

def evaluate_and_store(name, y_true, y_pred):
    from sklearn.metrics import precision_recall_fscore_support, accuracy_score

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='weighted'
    )
    accuracy = accuracy_score(y_true, y_pred)

    results.append({
        "Experiment": name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1
    })

# 🔥 BERT Experiments
# Use the already computed y_true and y_pred from the full fine-tuning
evaluate_and_store("Full BERT Fine-Tuning", y_true_full_bert, y_pred_full_bert)
evaluate_and_store("Frozen BERT", y_true_frozen_bert, y_pred_frozen_bert)
evaluate_and_store("2-Layer BERT Classifier", y_true_2layer_bert, y_pred_2layer_bert)

# DataFrame
import pandas as pd
df_results = pd.DataFrame(results)
print(df_results)

📝 Analysis & Insights


1. Full Fine-Tuning
Achieved the highest accuracy and F1 score
Model learns deep contextual features
However, training time is higher
2. Frozen BERT
Lowest performance among all experiments
Only classifier is trained → limited learning
Fastest training but poor generalization
3. Last 2 Layers Fine-Tuning
Balanced performance
Better than frozen model
Faster than full fine-tuning
Good trade-off between accuracy and efficiency